# Publication copy
The code and analytical commentary are preserved. Stored execution outputs were cleared to avoid publishing respondent previews or local execution details. Vetted aggregate tables and figures are provided in `course_artifacts`; see the repository results guide. The fully executed originals remain in the local course workspace. This notebook needs the separately acquired survey data and protected fit caches for a complete replay. No models were refitted for publication.


# Week 3 — Preparing the 2023 analysis dataset

This notebook completes the Week 3 data-preparation decisions and uses the **development countries only** for the first clustering experiments.

It has four roles:

1. **Curate the 13 profile features:** validate the raw survey codes, recode the six worry variables, and construct the seven respondent-or-known-person harm indicators.
2. **Freeze geographic validation:** reuse one fixed country-level split into development and held-out countries, with the exact country identities saved to disk.
3. **Document analysis eligibility and weighting:** audit development-set missingness, preserve incomplete respondents in the frozen data, exclude only respondents with no observed profile features from clustering, and construct equal-country analysis weights.
4. **Create Week 3 clustering candidates:** fit a simple K-means baseline for `k = 2,...,10`, assess reproducibility across development-country splits with Prediction Strength, and check whether selected solutions reduce mainly to overall burden or a single survey feature.

The held-out-country data are written and verified during data preparation, but **Section 11 does not load or use them for model development**. Median imputation and scaling are learned from development data, and are refitted inside the country-block Prediction Strength evaluations.


## 1. Set up a reproducible working environment


**What this cell does:** imports the required libraries, establishes display settings, locates the source files, creates the artifact folder, and computes SHA-256 hashes for the raw data and data dictionary.

**Why it matters:** fixed paths and immediate file checks prevent the notebook from silently using the wrong dataset. The hashes give the frozen country split a concrete source-data identity: if the source file later changes, the notebook can detect that rather than silently reusing an old split against different data.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

ROOT = Path.cwd()
if not (ROOT / "23_wrp" / "WRP_2023.csv").exists() and (ROOT.parent / "23_wrp" / "WRP_2023.csv").exists():
    ROOT = ROOT.parent

RAW_DATA = ROOT / "23_wrp" / "WRP_2023.csv"
DICTIONARY = ROOT / "23_wrp" / "WRP_2023_data_dictionary.csv"
ARTIFACTS = ROOT / "course_artifacts"
ARTIFACTS.mkdir(exist_ok=True)

assert RAW_DATA.exists(), f"Cannot find {RAW_DATA}"
assert DICTIONARY.exists(), f"Cannot find {DICTIONARY}"


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


raw_data_sha256 = sha256_file(RAW_DATA)
dictionary_sha256 = sha256_file(DICTIONARY)

print("Raw data SHA-256:", raw_data_sha256)
print("Dictionary SHA-256:", dictionary_sha256)


## 2. Declare the feature contract before touching the data


**What this cell does:** maps the opaque questionnaire codes to readable analysis names and fixes the 13 variables used for profile discovery: six ordered worry variables and seven binary harm-exposure variables.

**Why it matters:** feature selection is a scientific decision. Declaring the feature contract before clustering prevents variables from being added or removed in response to the clustering results. Demographic and contextual variables remain available for later objectives but do not define the Week 3 profiles.


In [ ]:
WORRY_FEATURES = {
    "WP20720": "worry_food",
    "WP20721": "worry_water",
    "WP20722": "worry_crime",
    "WP20723": "worry_severe_weather",
    "WP22213": "worry_traffic",
    "WP20726": "worry_mental_health",
}
EXPERIENCE_FEATURES = {
    "WP22442": "harm_food",
    "WP22443": "harm_water",
    "WP22444": "harm_crime",
    "WP22445": "harm_severe_weather",
    "WP22446": "harm_traffic",
    "WP22447": "harm_mental_health",
    "WP22448": "harm_work",
}
PROFILE_FEATURES = list(WORRY_FEATURES.values()) + list(EXPERIENCE_FEATURES.values())
assert len(PROFILE_FEATURES) == 13

display(pd.DataFrame({
    "raw_variable": list(WORRY_FEATURES) + list(EXPERIENCE_FEATURES),
    "analysis_feature": PROFILE_FEATURES,
    "role": ["ordered worry"] * 6 + ["binary harm"] * 7,
}))

## 3. Load the harmonised source table without discarding variables or respondents


**What this cell does:** loads the complete harmonised 2023 table, checks respondent and country identifiers, and declares the main variable families used in this week's work.

**Why it matters:** no respondent is discarded during source loading or recoding simply because one or more profile responses are missing. The full source record also preserves the demographic, socioeconomic, resilience, climate, workplace, and other variables needed later. Profile discovery itself remains restricted to the 13 features declared in Section 2.

Rows with incomplete profile information remain in the frozen dataset. Only later, immediately before clustering, respondents with **zero observed values across all 13 profile features** are excluded from profile assignment.

Uppercase names denote fixed feature lists and configuration choices. Lowercase names denote data objects created while the notebook runs.


In [ ]:
IDENTIFIER_COLUMNS = [
    "WPID_RANDOM", "Country", "COUNTRY_ISO3", "GlobalRegion",
    "CountryIncomeLevel", "WGT", "PROJWT",
]
BACKGROUND_FEATURES = [
    "Gender", "Age", "Education", "Urbanicity", "WP1223",
    "EMP_2010", "INCOME_5", "HouseholdSize", "ChildrenInHousehold",
]
PROFILE_SENSITIVITY_FEATURES = ["WP20719", "WP22214"]
RESILIENCE_FEATURES = [
    "resilience_index", "resilience_idv", "resilience_hhl",
    "resilience_com", "resilience_soc",
]
RAW_PROFILE_FEATURES = list(WORRY_FEATURES) + list(EXPERIENCE_FEATURES)

raw = pd.read_csv(RAW_DATA, low_memory=False)
print(f"Loaded {len(raw):,} respondents and all {raw.shape[1]} harmonised raw columns.")

assert raw["WPID_RANDOM"].is_unique, "Respondent identifiers must be unique."
assert raw["COUNTRY_ISO3"].notna().all(), "Every respondent needs a country identifier."
assert raw["COUNTRY_ISO3"].nunique() == 142
expected_columns = (
    IDENTIFIER_COLUMNS
    + BACKGROUND_FEATURES
    + PROFILE_SENSITIVITY_FEATURES
    + RESILIENCE_FEATURES
    + RAW_PROFILE_FEATURES
)
assert set(expected_columns).issubset(raw.columns)

## 4. Audit the raw response codes before recoding


**What this cell does:** converts the 13 raw profile columns to numeric form, checks every observed value against the documented response codes, and stops if a non-numeric or unexpected code is found. Only after this validation does it display the response-code audit.

**Why it matters:** unexpected survey values must not disappear into missingness during recoding. The raw-code contract is checked before `98` and `99` are converted to missing values.

### Codes for the six worry variables

- `1` = **very worried** that the risk could cause serious harm;
- `2` = **somewhat worried**;
- `3` = **not worried**;
- `98` = **don't know**;
- `99` = **refused to answer**.

Code `4` is not part of the worry scale and would trigger an error.

### Codes for the seven harm variables

- `1` = the **respondent personally** experienced serious harm;
- `2` = **someone personally known** by the respondent experienced serious harm;
- `3` = **both** the respondent and someone personally known experienced serious harm;
- `4` = **neither** the respondent nor someone personally known experienced serious harm;
- `98` = **don't know**;
- `99` = **refused to answer**.

The binary harm indicators constructed later combine codes `1`, `2`, and `3`. They therefore represent **reported harm to the respondent or someone personally known**, not personal harm to the respondent alone.


In [ ]:
WORRY_ALLOWED_RAW_CODES = {1, 2, 3, 98, 99}
HARM_ALLOWED_RAW_CODES = {1, 2, 3, 4, 98, 99}

raw_profile_numeric = pd.DataFrame(index=raw.index)
validation_rows = []

for source in RAW_PROFILE_FEATURES:
    numeric = pd.to_numeric(raw[source], errors="coerce")

    non_numeric_mask = raw[source].notna() & numeric.isna()
    if non_numeric_mask.any():
        examples = raw.loc[non_numeric_mask, source].astype(str).unique()[:10]
        raise ValueError(
            f"{source} contains non-numeric response values: {examples.tolist()}"
        )

    allowed_codes = (
        WORRY_ALLOWED_RAW_CODES
        if source in WORRY_FEATURES
        else HARM_ALLOWED_RAW_CODES
    )
    unexpected = sorted(
        set(numeric.dropna().unique().tolist()) - allowed_codes
    )
    if unexpected:
        raise ValueError(
            f"{source} contains unexpected survey codes: {unexpected}"
        )

    raw_profile_numeric[source] = numeric
    validation_rows.append({
        "raw_variable": source,
        "question_family": "worry" if source in WORRY_FEATURES else "harm",
        "observed_nonmissing": int(numeric.notna().sum()),
        "unexpected_codes": 0,
    })

raw_code_validation = pd.DataFrame(validation_rows)
display(raw_code_validation)

raw_code_audit = pd.concat(
    {
        source: raw_profile_numeric[source]
        .value_counts(dropna=False)
        .sort_index()
        for source in RAW_PROFILE_FEATURES
    },
    axis=1,
).T

# Validation above guarantees that reindexing cannot hide unexpected codes.
raw_code_audit = raw_code_audit.reindex(
    columns=[1, 2, 3, 4, 98, 99],
    fill_value=0,
)
raw_code_audit["missing"] = raw_profile_numeric.isna().sum()
raw_code_audit = raw_code_audit.fillna(0).astype(int)

dictionary_labels = (
    pd.read_csv(DICTIONARY)
    .set_index("variable")["label"]
    .to_dict()
)
raw_code_audit.insert(
    0,
    "meaning",
    [
        dictionary_labels.get(source, "Label not found")
        for source in raw_code_audit.index
    ],
)
display(raw_code_audit)


## 5. Recode worry so that larger values consistently mean greater worry


**What this cell does:** converts the validated worry responses from `1 = very worried`, `2 = somewhat worried`, and `3 = not worried` to `2`, `1`, and `0`. Codes `98` and `99` become missing.

**Why it matters:** all six worry variables then point in the same intuitive direction: larger values mean greater worry. The original ordinal ordering is preserved. The code does not treat special-response codes as extreme values, and unexpected raw codes have already been rejected in Section 4.

The resulting `0,1,2` representation is the Week 3 baseline. Using it with Euclidean distance later implies equal spacing between adjacent categories; that geometric assumption is explicitly left open for Week 4 sensitivity analysis.


In [ ]:
prepared = raw.copy()

for source, target in WORRY_FEATURES.items():
    values = raw_profile_numeric[source].mask(
        raw_profile_numeric[source].isin([98, 99])
    )
    # Raw coding: 1 = very worried, 2 = somewhat worried, 3 = not worried.
    # Analysis coding: 0 = not worried, 1 = somewhat worried, 2 = very worried.
    prepared[target] = values.map({1: 2.0, 2: 1.0, 3: 0.0})

display(prepared[list(WORRY_FEATURES.values())].describe().T)


## 6. Define and construct the binary harm-exposure indicators


**What this cell does:** maps raw harm codes `1`, `2`, and `3` to `1`, code `4` to `0`, and codes `98` and `99` to missing.

**Why it matters:** the resulting binary feature answers whether serious harm affected **the respondent, someone personally known, or both**. It does not distinguish direct personal harm from harm to someone known. The raw variables remain in the curated table so a narrower personal-only definition can be tested later if needed.

Throughout the Week 3 clustering analysis, “harm” therefore means **respondent-or-known-person harm exposure** unless stated otherwise.


In [ ]:
for source, target in EXPERIENCE_FEATURES.items():
    values = raw_profile_numeric[source].mask(
        raw_profile_numeric[source].isin([98, 99])
    )
    # 1 = respondent, 2 = someone known, 3 = both, 4 = neither.
    # The 2023 baseline uses respondent-or-known-person harm exposure.
    prepared[target] = values.map({1: 1.0, 2: 1.0, 3: 1.0, 4: 0.0})

display(
    prepared[list(EXPERIENCE_FEATURES.values())]
    .mean()
    .rename("share reporting respondent-or-known-person harm")
    .to_frame()
)


## 7. Create the country-level development and held-out split


**What this cell does:** builds one internally consistent country registry and either:

- **reuses** the exact saved allocation in `week03_country_split.csv`, if it already exists; or
- on the first run only, searches 50,000 reproducible candidate allocations and freezes one set of 29 held-out countries.

The first-run search keeps complete countries together and chooses a held-out set whose marginal distributions of `GlobalRegion` and substantive `CountryIncomeLevel` are close to those of the full 142-country registry. It does not use worry, harm, future outcomes, or cluster labels.

The code also checks that region and income classification are constant within each country. If a previously recorded raw-data hash exists and no longer matches the current source file, execution stops for review.

**Why it matters:** the external test is defined by the **identities of the held-out countries**, not only by an 80/20 ratio or a seed. Once the split is frozen, later notebook versions reuse the same country identities rather than searching again.

The balance score is recomputed from the actual frozen allocation on every run, whether the split was newly generated or reused.


In [ ]:
# A. Declare the split design and frozen split artifacts.
SPLIT_SEED = 2023
HELDOUT_SHARE = 0.20
N_CANDIDATE_SPLITS = 50_000

FROZEN_SPLIT_PATH = ARTIFACTS / "week03_country_split.csv"
SPLIT_DESIGN_PATH = ARTIFACTS / "week03_split_design.json"

# B. Country-level metadata must be internally consistent.
country_metadata_consistency = (
    raw.groupby("COUNTRY_ISO3")[
        ["GlobalRegion", "CountryIncomeLevel"]
    ]
    .nunique(dropna=False)
)
assert (
    country_metadata_consistency <= 1
).all().all(), "Region or income classification varies within a country."

country_registry = (
    raw.groupby("COUNTRY_ISO3", as_index=False)
    .agg(
        Country=("Country", "first"),
        GlobalRegion=("GlobalRegion", "first"),
        CountryIncomeLevel=("CountryIncomeLevel", "first"),
        respondents=("WPID_RANDOM", "size"),
    )
)

n_countries = len(country_registry)
requested_n_heldout = int(np.ceil(HELDOUT_SHARE * n_countries))

# C. Build the balance matrix.
substantive_income = country_registry["CountryIncomeLevel"].where(
    country_registry["CountryIncomeLevel"].isin([1, 2, 3, 4])
)
balance_matrix = pd.concat(
    [
        pd.get_dummies(
            country_registry["GlobalRegion"],
            prefix="region",
            dtype=int,
        ),
        pd.get_dummies(
            substantive_income,
            prefix="income",
            dtype=int,
        ),
    ],
    axis=1,
)
balance_values = balance_matrix.to_numpy()
overall_counts = balance_values.sum(axis=0)
target_heldout_counts = overall_counts * HELDOUT_SHARE

# D. Reuse the frozen allocation if it already exists.
split_reused = FROZEN_SPLIT_PATH.exists()

if split_reused:
    frozen_split = pd.read_csv(FROZEN_SPLIT_PATH)

    required_split_columns = {"COUNTRY_ISO3", "country_split"}
    assert required_split_columns.issubset(frozen_split.columns)
    assert frozen_split["COUNTRY_ISO3"].is_unique
    assert set(frozen_split["COUNTRY_ISO3"]) == set(
        country_registry["COUNTRY_ISO3"]
    )
    assert set(frozen_split["country_split"]).issubset(
        {"train", "heldout_country"}
    )

    if SPLIT_DESIGN_PATH.exists():
        previous_design = json.loads(
            SPLIT_DESIGN_PATH.read_text(encoding="utf-8")
        )
        previous_source_hash = previous_design.get("raw_data_sha256")
        if (
            previous_source_hash is not None
            and previous_source_hash != raw_data_sha256
        ):
            raise RuntimeError(
                "The raw-data checksum changed after the country split was "
                "frozen. Review the source data and split before continuing."
            )

    frozen_indexed = frozen_split.set_index("COUNTRY_ISO3")
    current_indexed = country_registry.set_index("COUNTRY_ISO3")
    for column in ["GlobalRegion", "CountryIncomeLevel"]:
        if column in frozen_indexed.columns:
            left = current_indexed[column].astype(str).sort_index()
            right = frozen_indexed[column].astype(str).sort_index()
            assert left.equals(right), (
                f"Frozen {column} metadata no longer matches the source."
            )

    split_map = frozen_indexed["country_split"]
    country_registry["country_split"] = (
        country_registry["COUNTRY_ISO3"].map(split_map)
    )
    split_source = "reused frozen country allocation"

else:
    # E. First run only: search candidate held-out sets.
    rng = np.random.default_rng(SPLIT_SEED)
    best_indices = None
    best_score = np.inf

    for _ in range(N_CANDIDATE_SPLITS):
        candidate_indices = rng.choice(
            n_countries,
            size=requested_n_heldout,
            replace=False,
        )
        candidate_counts = balance_values[candidate_indices].sum(axis=0)

        valid = (
            np.all(candidate_counts > 0)
            and np.all(candidate_counts < overall_counts)
        )
        if not valid:
            continue

        score = np.sum(
            (
                (candidate_counts - target_heldout_counts)
                / np.maximum(target_heldout_counts, 1)
            ) ** 2
        )
        if score < best_score:
            best_score = score
            best_indices = candidate_indices.copy()

    assert best_indices is not None, (
        "No valid balanced country split was found."
    )

    country_registry["country_split"] = "train"
    country_registry.loc[
        best_indices, "country_split"
    ] = "heldout_country"

    # Freeze the actual country identities immediately.
    country_registry.to_csv(FROZEN_SPLIT_PATH, index=False)
    split_source = "generated once and frozen"

# F. Audit the actual frozen allocation.
assert country_registry["country_split"].notna().all()

development_countries = set(
    country_registry.loc[
        country_registry["country_split"].eq("train"),
        "COUNTRY_ISO3",
    ]
)
heldout_countries = set(
    country_registry.loc[
        country_registry["country_split"].eq("heldout_country"),
        "COUNTRY_ISO3",
    ]
)

assert development_countries.isdisjoint(heldout_countries)
assert development_countries | heldout_countries == set(
    country_registry["COUNTRY_ISO3"]
)
assert len(heldout_countries) == requested_n_heldout

heldout_mask_registry = (
    country_registry["country_split"].eq("heldout_country").to_numpy()
)
heldout_counts = balance_values[heldout_mask_registry].sum(axis=0)
best_score = np.sum(
    (
        (heldout_counts - target_heldout_counts)
        / np.maximum(target_heldout_counts, 1)
    ) ** 2
)

region_balance = pd.crosstab(
    country_registry["GlobalRegion"],
    country_registry["country_split"],
)
income_balance = pd.crosstab(
    country_registry["CountryIncomeLevel"],
    country_registry["country_split"],
)
assert (region_balance > 0).all().all()
assert (
    income_balance.loc[[1, 2, 3, 4]] > 0
).all().all()

split_registry_sha256 = sha256_file(FROZEN_SPLIT_PATH)

# G. Attach the frozen assignment to respondents.
prepared = prepared.merge(
    country_registry[["COUNTRY_ISO3", "country_split"]],
    on="COUNTRY_ISO3",
    how="left",
    validate="many_to_one",
)
assert prepared["country_split"].notna().all()

display(
    prepared.groupby("country_split").agg(
        respondents=("WPID_RANDOM", "size"),
        countries=("COUNTRY_ISO3", "nunique"),
    )
)
display(Markdown("### Country counts by global region"))
display(region_balance)
display(Markdown("### Country counts by income level"))
display(income_balance)

print("Split source:", split_source)
print(f"Balance score = {best_score:.4f}")
print("Frozen split SHA-256:", split_registry_sha256)


### How to read the split diagnostics


The two displayed balance tables describe **countries**, not respondents. Exact equality is neither possible nor required because 29 whole countries must approximate the composition of a 142-country registry.

The checks require every observed global region and substantive income category (`1`–`4`) to occur in both the development and held-out sets. The balance score measures proportional discrepancies across those two margins. Income code `9` is retained in the registry but excluded from the balance objective because only one country has that code.

Respondent counts are deliberately not balanced. The purpose of this split is geographic transfer across countries, while respondent influence within the clustering is handled later through analysis weights.

The frozen held-out set tests transfer to additional countries drawn from broadly familiar regional and income contexts. It is not, by itself, a test of transfer to a completely unseen region, a future survey wave, or every possible national context.


## 8. Verify curated values and audit development-set missingness


**What this cell does:** first verifies the allowed values of the curated worry and harm features across the complete source table. It then restricts the **missingness summaries to development countries** and reports:

- missingness for each of the 13 profile features;
- how many profile features are observed for each development respondent;
- median and maximum feature-level missingness across development countries, including the country with the maximum for each feature.

**Why it matters:** value-range checks belong to data curation, whereas exploratory missingness summaries should not use the external held-out responses to guide modelling choices.

Missing values remain missing in the saved datasets. Respondents with all 13 profile features missing are still preserved at this stage; Section 11.1 excludes those rows from clustering so no profile is assigned entirely from imputed values.


In [ ]:
# A. Verify curated scales across the complete source table.
assert (
    prepared[list(WORRY_FEATURES.values())]
    .stack()
    .dropna()
    .isin([0, 1, 2])
    .all()
)
assert (
    prepared[list(EXPERIENCE_FEATURES.values())]
    .stack()
    .dropna()
    .isin([0, 1])
    .all()
)
assert prepared["WPID_RANDOM"].is_unique

# B. Inspect response missingness only in development countries.
development_for_quality = prepared.loc[
    prepared["country_split"].eq("train")
].copy()

quality = pd.DataFrame({
    "missing_n": development_for_quality[PROFILE_FEATURES].isna().sum(),
    "missing_pct": (
        100 * development_for_quality[PROFILE_FEATURES].isna().mean()
    ),
    "minimum": development_for_quality[PROFILE_FEATURES].min(),
    "maximum": development_for_quality[PROFILE_FEATURES].max(),
    "observed_value_count": (
        development_for_quality[PROFILE_FEATURES].nunique()
    ),
})
quality_display = quality.rename(columns={
    "missing_n": "Missing N",
    "missing_pct": "Missing (%)",
    "minimum": "Min",
    "maximum": "Max",
    "observed_value_count": "Distinct values",
})
display(Markdown("### Development feature-level quality"))
display(quality_display.round(2))

# C. How much profile information is observed for each development respondent?
development_observed_feature_count = (
    development_for_quality[PROFILE_FEATURES].notna().sum(axis=1)
)

respondent_observation_summary = pd.Series({
    "minimum observed profile features": int(
        development_observed_feature_count.min()
    ),
    "median observed profile features": float(
        development_observed_feature_count.median()
    ),
    "respondents with all 13 observed": int(
        development_observed_feature_count.eq(len(PROFILE_FEATURES)).sum()
    ),
    "respondents with zero observed": int(
        development_observed_feature_count.eq(0).sum()
    ),
    "respondents with <= 6 observed": int(
        development_observed_feature_count.le(6).sum()
    ),
}, name="development respondents")
display(Markdown("### Development respondent-level completeness"))
display(respondent_observation_summary.to_frame())

# D. Check whether missingness is concentrated in particular development countries.
development_country_missing_pct = (
    100
    * development_for_quality
    .groupby("COUNTRY_ISO3")[PROFILE_FEATURES]
    .agg(lambda series: series.isna().mean())
)

country_missingness_summary = pd.DataFrame({
    "median_country_missing_pct": development_country_missing_pct.median(),
    "max_country_missing_pct": development_country_missing_pct.max(),
    "country_with_max_missing": development_country_missing_pct.idxmax(),
})

country_missingness_display = country_missingness_summary.rename(
    columns={
        "median_country_missing_pct": "Median country missing (%)",
        "max_country_missing_pct": "Maximum country missing (%)",
        "country_with_max_missing": "Country at maximum",
    }
)
display(Markdown("### Development country-level missingness"))
display(country_missingness_display.round(2))


## 9. Construct equal-country analysis weights


### Why survey respondents do not necessarily represent equal numbers of people

The supplied `WGT` values are survey weights within each country. The notebook first checks that they are numeric, finite, and positive.

### Three weighting concepts in this project

- **`WGT`: within-country survey weight.** Preserves the survey designers' relative weighting of respondents inside a country.
- **`PROJWT`: population-proportional project weight.** Available in the source data for analyses where populous countries should contribute more to a pooled population estimate.
- **`weight_equal_country`: the Week 3 clustering weight.** Rescales `WGT` so that every country contributes the same total analysis weight within its split.

For respondent *i* in country *c*:

\[
w^{equal-country}_{ic}
= \frac{WGT_{ic}}{\sum_{j \in c} WGT_{jc}}
  \times \frac{1}{\text{number of countries in the split}}.
\]

The first factor preserves relative respondent weights inside a country; the second gives every country the same total contribution. Consequently, equal-country weights sum to one separately within the development and held-out sets.

**Why it matters:** this defines an **equal-country estimand**. Cluster prevalence under these weights should not be interpreted as the percentage of all adults worldwide in a profile.

Later comparisons between ordinary respondent shares and equal-country-weighted shares describe **prevalence sensitivity only**. They do not establish that an unweighted K-means refit would recover the same centroids or cluster assignments.


In [ ]:
# Survey weights must be numeric, finite, and positive before they are used.
prepared["WGT"] = pd.to_numeric(prepared["WGT"], errors="raise")
assert np.isfinite(prepared["WGT"]).all()
assert (prepared["WGT"] > 0).all()

# Within each split, every country receives equal total mass while the survey
# weights preserve relative respondent weights inside that country.
country_weight_total = (
    prepared.groupby(["country_split", "COUNTRY_ISO3"])["WGT"]
    .transform("sum")
)
n_countries_in_split = (
    prepared.groupby("country_split")["COUNTRY_ISO3"]
    .transform("nunique")
)
prepared["weight_equal_country"] = (
    prepared["WGT"] / country_weight_total / n_countries_in_split
)

check = (
    prepared.groupby(
        ["country_split", "COUNTRY_ISO3"]
    )["weight_equal_country"]
    .sum()
)
display(check.groupby(level=0).agg(["min", "max", "sum"]))


## 10. Freeze and save the development and held-out datasets


**What this cell does:** writes three respondent-level parquet files—the complete curated table, the development subset, and the held-out-country subset—and saves the 13-feature manifest. It records the raw-data hash, dictionary hash, frozen split hash, split-design parameters, and the balance score in `week03_split_design.json`.

The country identities are **not generated here**. Section 7 already created or reused the frozen country registry; this section verifies its hash and then saves the respondent-level datasets derived from it.

The three parquet files are read back only to check write integrity and respondent ordering.

**Why it matters:** downstream modelling should reuse the exact frozen development/held-out assignment. “Development” is used rather than simply “training” because those countries support preprocessing, internal country-block validation, model comparison, and interpretation. The held-out countries are reserved for later external geographic validation.


In [ ]:
combined_path = ARTIFACTS / "week03_2023_analysis_data.parquet"
development_path = ARTIFACTS / "week03_development_data.parquet"
heldout_path = ARTIFACTS / "week03_heldout_country_data.parquet"

development_data = prepared.loc[
    prepared["country_split"].eq("train")
].copy()
heldout_country_data = prepared.loc[
    prepared["country_split"].eq("heldout_country")
].copy()

assert len(development_data) + len(heldout_country_data) == len(prepared)
assert set(development_data["COUNTRY_ISO3"]).isdisjoint(
    set(heldout_country_data["COUNTRY_ISO3"])
)
assert set(development_data["WPID_RANDOM"]).isdisjoint(
    set(heldout_country_data["WPID_RANDOM"])
)

prepared.to_parquet(combined_path, index=False)
development_data.to_parquet(development_path, index=False)
heldout_country_data.to_parquet(heldout_path, index=False)

manifest = pd.DataFrame({
    "feature": PROFILE_FEATURES,
    "role": (
        ["ordered worry"] * 6
        + ["binary respondent-or-known-person harm"] * 7
    ),
    "allowed_values": ["0, 1, 2"] * 6 + ["0, 1"] * 7,
    "used_for_profile_discovery": True,
})
manifest.to_csv(
    ARTIFACTS / "week03_profile_feature_manifest.csv",
    index=False,
)

assert FROZEN_SPLIT_PATH.exists()
assert sha256_file(FROZEN_SPLIT_PATH) == split_registry_sha256

split_design_record = {
    "split_unit": "country",
    "development_label_in_file": "train",
    "heldout_label_in_file": "heldout_country",
    "heldout_share_requested": HELDOUT_SHARE,
    "heldout_countries_selected": len(heldout_countries),
    "balance_dimensions": "GlobalRegion; CountryIncomeLevel",
    "candidate_allocations_searched_on_first_run": N_CANDIDATE_SPLITS,
    "random_seed_for_first_run_search": SPLIT_SEED,
    "selected_balance_score": float(best_score),
    "split_source_this_run": split_source,
    "raw_data_sha256": raw_data_sha256,
    "dictionary_sha256": dictionary_sha256,
    "country_split_sha256": split_registry_sha256,
}
SPLIT_DESIGN_PATH.write_text(
    json.dumps(split_design_record, indent=2),
    encoding="utf-8",
)

combined_round_trip = pd.read_parquet(combined_path)
development_round_trip = pd.read_parquet(development_path)
heldout_round_trip = pd.read_parquet(heldout_path)

assert combined_round_trip.shape == prepared.shape
assert development_round_trip["WPID_RANDOM"].equals(
    development_data["WPID_RANDOM"].reset_index(drop=True)
)
assert heldout_round_trip["WPID_RANDOM"].equals(
    heldout_country_data["WPID_RANDOM"].reset_index(drop=True)
)

saved_files = pd.DataFrame({
    "artifact": ["combined", "development", "held-out countries"],
    "file": [
        combined_path.name,
        development_path.name,
        heldout_path.name,
    ],
    "respondents": [
        len(prepared),
        len(development_data),
        len(heldout_country_data),
    ],
    "countries": [
        prepared["COUNTRY_ISO3"].nunique(),
        development_data["COUNTRY_ISO3"].nunique(),
        heldout_country_data["COUNTRY_ISO3"].nunique(),
    ],
})
display(saved_files.rename(columns={
    "artifact": "Artifact",
    "file": "File",
    "respondents": "N",
    "countries": "Countries",
}))


## 11. Begin Objective 1 with a development-only K-means baseline


The data-curation decisions and country allocation are now frozen. Section 11 removes the respondent-level preparation objects, reloads **only the saved development file**, and performs the Week 3 clustering analysis without opening the held-out-country parquet file.

The baseline analysis does the following:

- excludes only development respondents with **zero observed values across all 13 profile features**;
- uses the 13 declared profile features and no demographic/context variables;
- median-imputes the remaining missing values and z-standardizes the features on the development data;
- fits equal-country-weighted K-means for `k = 2,...,10`;
- reports cluster-size, compactness, separation, explained-dispersion, and initialization-stability diagnostics;
- uses repeated country-block Prediction Strength as the **primary Week 3 reproducibility rule**, retaining every `k` whose mean Prediction Strength is at least `0.75`;
- for the retained resolutions, tests whether the partition is largely explained by two overall burden scores or whether individual clusters are almost defined by one survey feature;
- describes the retained clusters using the original non-imputed response distributions.

This is still a **baseline representation**. Week 4 will challenge the surviving candidates with alternative scaling/feature geometries, clustering algorithms, imputation choices, and feature/domain sensitivity analyses.

No PCA is used, and nothing in Section 11 reads the held-out-country parquet file.


### 11.1 Reset the workspace and reload development data from disk


**What this cell does:** retains only scalar checkpoints from data preparation, removes respondent-level preparation objects from memory, and reloads the frozen development parquet using only respondent ID, country ID, equal-country weight, and the 13 profile features.

It then counts observed profile features per respondent. Rows with **zero observed profile features** are excluded from the clustering dataframe; all other incomplete rows remain eligible for later imputation. The code also checks that this exclusion does not remove an entire development country.

**Why it matters:** the modelling section starts from a visible development-only boundary. The held-out-country file is not opened, and no respondent is assigned a cluster purely from imputed values.


In [ ]:
# Keep scalar checkpoints from the preparation stage before clearing
# respondent-level preparation objects.
TOTAL_RESPONDENTS = len(prepared)
DEVELOPMENT_COUNTRIES = len(development_countries)
HELDOUT_COUNTRIES = len(heldout_countries)

objects_to_remove = [
    "raw",
    "raw_profile_numeric",
    "raw_code_audit",
    "raw_code_validation",
    "prepared",
    "development_data",
    "heldout_country_data",
    "development_for_quality",
    "development_country_missing_pct",
    "combined_round_trip",
    "development_round_trip",
    "heldout_round_trip",
    "country_registry",
    "region_balance",
    "income_balance",
    "saved_files",
    "development_countries",
    "heldout_countries",
    "best_indices",
    "frozen_split",
]
for object_name in objects_to_remove:
    globals().pop(object_name, None)

development_columns = [
    "WPID_RANDOM",
    "COUNTRY_ISO3",
    "country_split",
    "weight_equal_country",
] + PROFILE_FEATURES

development_data = pd.read_parquet(
    ARTIFACTS / "week03_development_data.parquet",
    columns=development_columns,
)

assert development_data["country_split"].eq("train").all()
assert (
    development_data["COUNTRY_ISO3"].nunique()
    == DEVELOPMENT_COUNTRIES
)
assert set(PROFILE_FEATURES).issubset(development_data.columns)

development_data = development_data.drop(columns="country_split")

development_observed_count = (
    development_data[PROFILE_FEATURES].notna().sum(axis=1)
)
fully_missing_profile = development_observed_count.eq(0)
EXCLUDED_FULLY_MISSING_PROFILES = int(
    fully_missing_profile.sum()
)

if EXCLUDED_FULLY_MISSING_PROFILES:
    development_data = (
        development_data.loc[~fully_missing_profile]
        .reset_index(drop=True)
    )

assert (
    development_data["COUNTRY_ISO3"].nunique()
    == DEVELOPMENT_COUNTRIES
), "Removing fully unobserved rows removed an entire development country."

assert list(development_data.columns) == [
    "WPID_RANDOM",
    "COUNTRY_ISO3",
    "weight_equal_country",
    *PROFILE_FEATURES,
]

print(
    f"Reloaded {len(development_data):,} profile-eligible development "
    f"respondents from "
    f"{development_data['COUNTRY_ISO3'].nunique()} countries."
)
print(
    "Excluded from clustering because all 13 profile features were missing:",
    EXCLUDED_FULLY_MISSING_PROFILES,
)
print("The held-out-country file has not been loaded in this section.")

display(pd.Series({
    "profile-eligible respondents": len(development_data),
    "excluded fully unobserved respondents": EXCLUDED_FULLY_MISSING_PROFILES,
    "countries": development_data["COUNTRY_ISO3"].nunique(),
    "profile features": len(PROFILE_FEATURES),
}, name="development-only baseline"))


### 11.2 Fit development-only preprocessing


**What this cell does:** fits the Week 3 preprocessing baseline on all profile-eligible development respondents:

1. median imputation for each of the 13 features;
2. ordinary, unweighted z-standardization using `StandardScaler`.

It also draws **one fixed random sample of up to 10,000 development respondents** using seed `2023`. That same sample is later used for the unweighted silhouette, Calinski--Harabasz, and Davies--Bouldin calculations for every candidate `k`.

**Why it matters:** K-means is fitted later on the **full profile-eligible development dataset**, not on the 10,000-person diagnostic sample. The subsample is used only for those three internal-validity indices. In practice, silhouette is the main computational reason for sampling; using the same rows for all three indices keeps those diagnostic calculations on a common evaluation set.

The scaler itself is fitted without the equal-country sample weights, while K-means later uses those weights. This is intentional for the Week 3 baseline rather than a claim that unweighted z-standardization is uniquely correct. Week 4 will explicitly test weighted standardization and a range/block-balanced representation.

The code also verifies before imputation that every profile feature has at least one observed value in the development data. Prediction Strength repeats that check inside every training-country half.


In [ ]:
from itertools import combinations

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

KMEANS_SEED = 2023
CANDIDATE_K = range(2, 11)
KMEANS_N_INIT = 30
QUALITY_SAMPLE = 10_000
STABILITY_SEEDS = [11, 29, 47, 71]
STABILITY_N_INIT = 5

development_features = development_data[PROFILE_FEATURES]

assert development_features.notna().any(axis=0).all(), (
    "Every profile feature must have at least one observed development value."
)

profile_imputer = SimpleImputer(strategy="median")
profile_scaler = StandardScaler()

development_imputed = profile_imputer.fit_transform(development_features)
development_scaled = profile_scaler.fit_transform(development_imputed)

assert development_scaled.shape == (len(development_data), len(PROFILE_FEATURES))
assert not np.isnan(development_scaled).any()

quality_rng = np.random.default_rng(KMEANS_SEED)
quality_indices = quality_rng.choice(
    len(development_data),
    size=min(QUALITY_SAMPLE, len(development_data)),
    replace=False,
)
quality_scaled = development_scaled[quality_indices]

preprocessing_summary = pd.DataFrame({
    "feature": PROFILE_FEATURES,
    "imputation_median": profile_imputer.statistics_,
    "mean_after_imputation": profile_scaler.mean_,
    "standard_deviation": profile_scaler.scale_,
})
preprocessing_display = preprocessing_summary.rename(columns={
    "feature": "Feature",
    "imputation_median": "Impute median",
    "mean_after_imputation": "Mean",
    "standard_deviation": "Std. dev.",
})
display(preprocessing_display.round(3))

print(
    f"Internal-validity measures will use the same reproducible sample of "
    f"{len(quality_indices):,} development respondents."
)


### 11.3 Fit and compare `k = 2` through `k = 10`


**What this cell does:** for each `k = 2,...,10`, fits K-means on the **full standardized development dataset** with equal-country respondent weights and stores the fitted model and cluster labels.

It then calculates four groups of diagnostics.

### Cluster-size and prevalence diagnostics

- **cluster-size imbalance:** half the summed absolute deviation of the equal-country-weighted cluster shares from the equal-size benchmark `1/k`. Lower values indicate more balanced cluster sizes.
- **respondent vs equal-country prevalence gap:** the largest absolute percentage-point difference between ordinary respondent shares and equal-country-weighted shares for corresponding fitted clusters.

The prevalence gap is retained as a descriptive check but is **not a clustering-quality criterion**. It compares two ways of counting the **same weighted-K-means assignments**; it does not compare weighted and unweighted K-means models. Because the full prevalence tables for retained solutions are shown in Section 11.6, this diagnostic is no longer given its own plot in Section 11.4.

### Weighted compactness diagnostics

- **inertia:** the fitted model's equal-country-weighted within-cluster squared distance; lower means more compact clusters, although inertia necessarily falls as `k` increases;
- **incremental inertia improvement:** the percentage decrease in inertia when moving from `k-1` to `k`; a larger value means that the additional cluster produced a larger compactness gain.

The code also stores **explained weighted dispersion**, calculated as `1 - inertia / total weighted dispersion`. This is a normalized restatement of inertia on the same fixed dataset: as inertia decreases, explained dispersion increases. Because it contains no independent information for choosing `k`, it remains in the exported metrics but is no longer emphasized in the notebook table or given a separate plot.

### Unweighted internal-validity indices

Silhouette, Calinski--Harabasz, and Davies--Bouldin are calculated from the **same fixed sample of up to 10,000 development respondents** selected in Section 11.2. They are not weighted. The K-means model and labels from which those sampled labels are taken were nevertheless fitted on the full development dataset with equal-country weights.

### Initialization stability

For each `k`, four additional full-development K-means fits are run with seeds `11`, `29`, `47`, and `71`, using `n_init = 5`. The reported stability value is the **mean pairwise adjusted Rand index among those four label sets**. Higher values mean that different initializations recover more similar partitions.

**Why it matters:** K-means will produce a partition for every requested `k`, so successful fitting is not evidence that a useful profile structure exists. These diagnostics describe size, compactness, separation, and sensitivity to initialization. They do not by themselves select `k`; the primary reproducibility decision comes from country-block Prediction Strength in Section 11.5.


In [ ]:
baseline_models = {}
baseline_metric_rows = []
baseline_cluster_size_tables = []

analysis_weights = development_data["weight_equal_country"].to_numpy()
weighted_center = np.average(development_scaled, axis=0, weights=analysis_weights)
weighted_total_dispersion = np.sum(
    analysis_weights[:, None] * (development_scaled - weighted_center) ** 2
)

for k in CANDIDATE_K:
    model = KMeans(n_clusters=k, n_init=KMEANS_N_INIT, random_state=KMEANS_SEED)
    labels = model.fit_predict(
        development_scaled,
        sample_weight=analysis_weights,
    )

    respondent_counts = np.bincount(labels, minlength=k)
    equal_country_weight = (
        pd.Series(analysis_weights)
        .groupby(labels)
        .sum()
        .reindex(range(k), fill_value=0)
    )

    respondent_pct = 100 * respondent_counts / respondent_counts.sum()
    equal_country_weighted_pct = 100 * equal_country_weight / equal_country_weight.sum()
    expected_cluster_pct = 100 / k

    cluster_sizes = pd.DataFrame({
        "k": k,
        "cluster": range(k),
        "respondents": respondent_counts,
        "respondent_pct": respondent_pct,
        "equal_country_weighted_pct": equal_country_weighted_pct,
    })
    baseline_cluster_size_tables.append(cluster_sizes)

    quality_labels = labels[quality_indices]

    stability_label_sets = []
    for stability_seed in STABILITY_SEEDS:
        stability_model = KMeans(
            n_clusters=k,
            n_init=STABILITY_N_INIT,
            random_state=stability_seed,
        )
        stability_label_sets.append(
            stability_model.fit_predict(
                development_scaled,
                sample_weight=analysis_weights,
            )
        )

    pairwise_ari = [
        adjusted_rand_score(labels_a, labels_b)
        for labels_a, labels_b in combinations(stability_label_sets, 2)
    ]

    baseline_metric_rows.append({
        "k": k,
        "cluster_size_imbalance_pct": 0.5 * np.abs(
            equal_country_weighted_pct.to_numpy() - expected_cluster_pct
        ).sum(),
        "max_weighting_share_gap_pct_points": np.max(
            np.abs(respondent_pct - equal_country_weighted_pct.to_numpy())
        ),
        "inertia": model.inertia_,
        "explained_dispersion_pct": 100 * (1 - model.inertia_ / weighted_total_dispersion),
        "silhouette": silhouette_score(quality_scaled, quality_labels),
        "calinski_harabasz": calinski_harabasz_score(quality_scaled, quality_labels),
        "davies_bouldin": davies_bouldin_score(quality_scaled, quality_labels),
        "initialization_stability_ari": np.mean(pairwise_ari),
    })
    baseline_models[k] = {"model": model, "labels": labels}

baseline_metrics = pd.DataFrame(baseline_metric_rows).set_index("k")
baseline_metrics["incremental_inertia_improvement_pct"] = (
    -100 * baseline_metrics["inertia"].pct_change()
)

baseline_cluster_sizes = pd.concat(baseline_cluster_size_tables, ignore_index=True)

sanity_check_columns = [
    "cluster_size_imbalance_pct",
    "max_weighting_share_gap_pct_points",
]

# Display only diagnostics that add distinct information.
# explained_dispersion_pct is still stored in baseline_metrics and exported,
# but it is a monotonic rescaling of inertia for this fixed dataset.
quality_metric_columns = [
    "inertia",
    "incremental_inertia_improvement_pct",
    "silhouette",
    "calinski_harabasz",
    "davies_bouldin",
    "initialization_stability_ari",
]

sanity_display = baseline_metrics[sanity_check_columns].copy().round(3)
sanity_display.columns = pd.MultiIndex.from_tuples([
    ("Size imbalance", "%"),
    ("Weighting gap", "pp"),
])

quality_display = baseline_metrics[quality_metric_columns].copy().round(3)
quality_display.columns = pd.MultiIndex.from_tuples([
    ("Inertia", ""),
    ("Inertia gain", "%"),
    ("Silhouette", ""),
    ("CH index", ""),
    ("DB index", ""),
    ("Stability", "ARI"),
])

cluster_sizes_display = baseline_cluster_sizes.copy().round(3)
cluster_sizes_display.columns = pd.MultiIndex.from_tuples([
    ("k", ""),
    ("Cluster", ""),
    ("N", ""),
    ("Respondents", "%"),
    ("Equal-country", "%"),
])
display(Markdown("#### Simple cluster-size sanity checks"))
display(sanity_display)

display(Markdown("#### Internal-quality diagnostics"))
display(quality_display)

display(Markdown("#### Complete respondent and equal-country-weighted cluster sizes"))
display(cluster_sizes_display)


### 11.4 Visualize the sanity checks first, then the quality trade-offs


**What this cell does:** visualizes the most useful Week 3 diagnostics without giving every stored quantity its own figure.

The first figure shows **cluster-size imbalance**. Lower is better: `0%` would mean equal-country-weighted clusters of exactly equal size. This is only a sanity check—real profiles do not need to be equal-sized—but very large imbalance can reveal a solution dominated by one broad group and several small fragments.

The former “respondents vs equal-country shares” plot has been removed. That quantity only asked whether the **prevalence of the already fitted clusters** changes when assignments are counted by respondents versus equal-country weights. It does not assess cluster quality and does not tell us whether an unweighted K-means fit would recover the same partition. The actual prevalence comparison remains available in the tables, especially Section 11.6 for the retained solutions.

The second figure contains six diagnostics:

- **weighted inertia — lower is better**, but it always decreases as more clusters are added;
- **gain from adding one cluster — higher means a larger incremental improvement**, so we look for where the gains start becoming small rather than simply maximizing it;
- **silhouette — higher is better**;
- **Calinski--Harabasz — higher is better**;
- **Davies--Bouldin — lower is better**;
- **initialization stability (ARI) — higher is better**.

The former standalone **weighted dispersion explained** plot has also been removed. On this fixed dataset it is just a normalized transformation of inertia, so it does not add an independent criterion for `k`. The numerical value remains stored in the exported metrics for completeness.

**Why it matters:** these plots make different aspects of the candidate solutions easier to interpret, but they remain supporting diagnostics. Country-block Prediction Strength in Section 11.5 is the primary Week 3 reproducibility rule.


In [ ]:
# --- Cluster-size sanity check ---
# This plot is intentionally kept separate from model-quality diagnostics.
# Equal cluster sizes are not a requirement, but extreme imbalance can signal
# that a candidate k is splitting off very small groups.

fig, ax = plt.subplots(figsize=(7, 4.5))

baseline_metrics["cluster_size_imbalance_pct"].plot(
    marker="o",
    ax=ax,
    color="#54A24B",
)

ax.axhline(0.0, color="grey", linestyle="--", linewidth=1)
ax.set(
    title="Cluster-size imbalance — lower is better",
    xlabel="Number of clusters, k",
    ylabel="Weight to redistribute (%)",
    xticks=list(CANDIDATE_K),
)
plt.tight_layout()
plt.show()

# The respondent-vs-equal-country prevalence gap is deliberately NOT plotted.
# It describes how the SAME fitted assignments are counted under two weighting
# views; it is not a clustering-quality measure. The numerical diagnostic is
# retained in baseline_metrics and the detailed prevalence tables are shown in
# Section 11.6 for the Prediction-Strength-retained solutions.


# --- Internal-quality diagnostics ---
# Direction-of-improvement is written directly into each title so students do
# not need to remember which indices are maximized and which are minimized.

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

baseline_metrics["inertia"].plot(
    marker="o",
    ax=axes[0, 0],
    color="#4C78A8",
)
baseline_metrics["incremental_inertia_improvement_pct"].plot(
    marker="o",
    ax=axes[0, 1],
    color="#72B7B2",
)
baseline_metrics["silhouette"].plot(
    marker="o",
    ax=axes[0, 2],
    color="#F58518",
)
baseline_metrics["calinski_harabasz"].plot(
    marker="o",
    ax=axes[1, 0],
    color="#B279A2",
)
baseline_metrics["davies_bouldin"].plot(
    marker="o",
    ax=axes[1, 1],
    color="#FF9DA6",
)
baseline_metrics["initialization_stability_ari"].plot(
    marker="o",
    ax=axes[1, 2],
    color="#59A14F",
)

axes[0, 0].set(
    title="Weighted inertia — lower is better",
    ylabel="Weighted within-cluster squared distance",
)
axes[0, 1].set(
    title="Gain from adding one cluster — higher = larger gain",
    ylabel="Inertia reduction (%)",
)
axes[0, 2].set(
    title="Silhouette — higher is better",
    ylabel="Silhouette score",
)
axes[1, 0].set(
    title="Calinski--Harabasz — higher is better",
    ylabel="CH index",
)
axes[1, 1].set(
    title="Davies--Bouldin — lower is better",
    ylabel="DB index",
)
axes[1, 2].set(
    title="Initialization stability — higher is better",
    ylabel="Mean pairwise ARI",
)

for axis in axes.flat:
    axis.set(
        xlabel="Number of clusters, k",
        xticks=list(CANDIDATE_K),
    )

plt.tight_layout()
plt.show()

# explained_dispersion_pct is intentionally not plotted. For this fixed
# development matrix it is a normalized transformation of weighted inertia and
# therefore supplies no independent evidence for choosing k.


### 11.5 Identify reproducible solutions with country-block Prediction Strength


Prediction Strength is the primary Week 3 rule for identifying reproducible clustering resolutions.

### Why repeated two-half country splits?

The `2-fold` terminology can be misleading here because the purpose is not ordinary predictive cross-validation. Each repeat deliberately divides the **113 development countries into two large, disjoint country halves**. A model is fitted on roughly half the countries and challenged to reproduce cluster co-membership in the other half.

This is a relatively demanding geographic stress test. Replacing it with ordinary 5-fold cross-validation would give each training fit about 80% of the countries and only about 20% to the validation side, making the transfer problem easier and leaving fewer countries for the independent comparison clustering.

To obtain a more stable estimate without weakening that stress test, this version uses **10 repeated two-half splits** instead of five. Both directions are evaluated in every repeat, giving **20 country-block evaluations per `k`**.

For each direction:

1. fit the median imputer and unweighted `StandardScaler` on the training-country half only;
2. fit equal-country-weighted K-means on that training half;
3. use those fitted centroids to predict labels in the other country half;
4. independently fit equal-country-weighted K-means in that other half;
5. for every independently fitted test-half cluster, calculate the fraction of respondent pairs that the training-derived model also places together;
6. take the **minimum** of those cluster-specific agreements as the Prediction Strength for that evaluation.

The working Week 3 rule remains:

> **Retain every `k` whose mean Prediction Strength across the 20 evaluations is at least `0.75`.**

The `0.75` threshold was chosen during exploratory development and is now fixed for subsequent comparisons.

### How to read the Prediction Strength plot

The plot now distinguishes the components explicitly:

- **small points:** the 20 individual country-half evaluations;
- **large point:** the mean Prediction Strength for that `k`;
- **vertical error bar:** mean ± one standard deviation;
- **dashed horizontal line:** the `0.75` working threshold.

The SD bar is a **variability summary, not a confidence interval and not a selection interval**. A value of `k` is therefore not retained merely because its SD bar or some individual evaluations extend above `0.75`. Occasional high Prediction Strength with a substantially lower mean indicates that the resolution works for some country partitions but is not consistently reproducible under the current rule.

Several values of `k` may survive. The finest surviving resolution is recorded descriptively, but it is not automatically declared the final profile solution.

Reference: Tibshirani R, Walther G. *Cluster Validation by Prediction Strength*. Journal of Computational and Graphical Statistics. 2005;14(3):511--528. doi:10.1198/106186005X59243.


In [ ]:
# Prediction Strength configuration.
# We keep a demanding repeated two-half COUNTRY split rather than switching
# to ordinary 5-fold CV. Each training model therefore sees only about half
# of the development countries. Ten repeats and both directions give
# 20 evaluations per k, which stabilizes the mean and variability summaries.
PS_THRESHOLD = 0.75
PS_REPEATS = 10
PS_N_INIT = 10
PS_SEED = 2023


def prediction_strength_from_labels(test_labels, predicted_labels, k):
    """Minimum pairwise co-membership agreement over independently found test clusters."""
    cluster_strengths = []

    for cluster in range(k):
        members = test_labels == cluster
        n_members = int(members.sum())

        if n_members < 2:
            cluster_strengths.append(0.0)
            continue

        predicted_counts = np.bincount(
            predicted_labels[members],
            minlength=k,
        )

        same_cluster_ordered_pairs = np.sum(
            predicted_counts * (predicted_counts - 1)
        )
        all_ordered_pairs = n_members * (n_members - 1)

        cluster_strengths.append(
            same_cluster_ordered_pairs / all_ordered_pairs
        )

    return min(cluster_strengths)


development_countries_for_ps = np.array(
    sorted(development_data["COUNTRY_ISO3"].unique())
)
ps_rng = np.random.default_rng(PS_SEED)
prediction_strength_rows = []

for repeat in range(PS_REPEATS):
    # Randomly divide the DEVELOPMENT COUNTRIES, not individual respondents,
    # into two nearly equal halves. This deliberately creates a hard transfer
    # task: each fitted model sees only about half of the country contexts.
    shuffled_countries = ps_rng.permutation(development_countries_for_ps)
    split_point = len(shuffled_countries) // 2

    country_half_a = set(shuffled_countries[:split_point])
    country_half_b = set(shuffled_countries[split_point:])

    # Evaluate both directions so each half serves once as training countries
    # and once as an independent country block on which reproducibility is tested.
    fold_directions = [
        ("A_to_B", country_half_a, country_half_b),
        ("B_to_A", country_half_b, country_half_a),
    ]

    for direction, train_countries, test_countries in fold_directions:
        train_mask = development_data["COUNTRY_ISO3"].isin(train_countries)
        test_mask = development_data["COUNTRY_ISO3"].isin(test_countries)

        train_features = development_data.loc[train_mask, PROFILE_FEATURES]
        test_features = development_data.loc[test_mask, PROFILE_FEATURES]

        assert train_features.notna().any(axis=0).all(), (
            "A Prediction Strength training fold has a completely "
            "unobserved profile feature."
        )

        # Learn ALL data-dependent preprocessing from the training-country
        # half only. The other country half is transformed with those fitted
        # objects so no information leaks across the geographic validation split.
        fold_imputer = SimpleImputer(strategy="median")
        fold_scaler = StandardScaler()

        train_imputed = fold_imputer.fit_transform(train_features)
        test_imputed = fold_imputer.transform(test_features)

        train_scaled = fold_scaler.fit_transform(train_imputed)
        test_scaled = fold_scaler.transform(test_imputed)

        train_weights = development_data.loc[
            train_mask, "weight_equal_country"
        ].to_numpy()
        test_weights = development_data.loc[
            test_mask, "weight_equal_country"
        ].to_numpy()

        for k in CANDIDATE_K:
            fold_seed = PS_SEED + 1000 * repeat + 100 * k + (
                1 if direction == "A_to_B" else 2
            )

            # Model 1: learn clusters from the training-country half, then
            # assign the independent country half to those learned centroids.
            train_model = KMeans(
                n_clusters=k,
                n_init=PS_N_INIT,
                random_state=fold_seed,
            )
            train_model.fit(
                train_scaled,
                sample_weight=train_weights,
            )
            predicted_test_labels = train_model.predict(test_scaled)

            # Model 2: cluster the other country half independently. Prediction
            # Strength asks whether pairs that belong together in these
            # independently discovered clusters are also kept together by the
            # model transported from the training countries.
            independent_test_model = KMeans(
                n_clusters=k,
                n_init=PS_N_INIT,
                random_state=fold_seed + 50_000,
            )
            independent_test_labels = independent_test_model.fit_predict(
                test_scaled,
                sample_weight=test_weights,
            )

            # The function returns the WORST cluster-specific pair agreement.
            # A high score therefore requires every independent test cluster to
            # be reasonably reproducible, not just the largest/easiest cluster.
            fold_prediction_strength = prediction_strength_from_labels(
                independent_test_labels,
                predicted_test_labels,
                k,
            )

            prediction_strength_rows.append({
                "repeat": repeat + 1,
                "direction": direction,
                "k": k,
                "prediction_strength": fold_prediction_strength,
                "train_countries": len(train_countries),
                "test_countries": len(test_countries),
                "train_respondents": int(train_mask.sum()),
                "test_respondents": int(test_mask.sum()),
            })

prediction_strength_folds = pd.DataFrame(prediction_strength_rows)

prediction_strength_summary = (
    prediction_strength_folds
    .groupby("k")["prediction_strength"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={
        "mean": "prediction_strength_mean",
        "std": "prediction_strength_sd",
        "min": "prediction_strength_min",
        "max": "prediction_strength_max",
        "count": "n_evaluations",
    })
)

fraction_above_threshold = (
    prediction_strength_folds
    .assign(
        above_threshold=lambda frame:
        frame["prediction_strength"] >= PS_THRESHOLD
    )
    .groupby("k")["above_threshold"]
    .mean()
)

prediction_strength_summary["evaluations_above_threshold_pct"] = (
    100 * fraction_above_threshold
)
prediction_strength_summary["passes_threshold"] = (
    prediction_strength_summary["prediction_strength_mean"] >= PS_THRESHOLD
)

selected_k_values = [
    int(k)
    for k in prediction_strength_summary.index[
        prediction_strength_summary["passes_threshold"]
    ].tolist()
]

finest_selected_k = max(selected_k_values) if selected_k_values else None

prediction_strength_display = (
    prediction_strength_summary[
        [
            "prediction_strength_mean",
            "prediction_strength_sd",
            "prediction_strength_min",
            "prediction_strength_max",
            "evaluations_above_threshold_pct",
            "n_evaluations",
            "passes_threshold",
        ]
    ]
    .copy()
    .round(3)
)
prediction_strength_display.columns = pd.MultiIndex.from_tuples([
    ("Prediction Strength", "mean"),
    ("Prediction Strength", "SD"),
    ("Prediction Strength", "min"),
    ("Prediction Strength", "max"),
    (f"Evaluations ≥ {PS_THRESHOLD:.2f}", "%"),
    ("CV evaluations", "N"),
    (f"Mean PS ≥ {PS_THRESHOLD:.2f}", ""),
])

display(prediction_strength_display)

# Visualize the distribution of country-half evaluations directly.
# Small points = individual evaluations.
# Large point + error bar = mean +/- 1 SD.
# The SD bar describes variability; it is NOT a confidence interval and does
# not change the mean-based selection rule.
fig, ax = plt.subplots(figsize=(9, 5.2))

for k in CANDIDATE_K:
    values = prediction_strength_folds.loc[
        prediction_strength_folds["k"].eq(k),
        "prediction_strength",
    ].to_numpy()

    # Deterministic horizontal offsets prevent the 20 evaluation points from
    # drawing exactly on top of one another.
    offsets = np.linspace(-0.14, 0.14, len(values))
    ax.scatter(
        k + offsets,
        values,
        s=22,
        alpha=0.45,
        label="Individual country-half evaluations" if k == min(CANDIDATE_K) else None,
    )

    mean_value = prediction_strength_summary.loc[
        k, "prediction_strength_mean"
    ]
    sd_value = prediction_strength_summary.loc[
        k, "prediction_strength_sd"
    ]

    ax.errorbar(
        k,
        mean_value,
        yerr=sd_value,
        fmt="o",
        markersize=7,
        capsize=4,
        linewidth=1.5,
        label="Mean +/- 1 SD" if k == min(CANDIDATE_K) else None,
    )

ax.axhline(
    PS_THRESHOLD,
    linestyle="--",
    linewidth=1.2,
    label=f"Working threshold = {PS_THRESHOLD:.2f}",
)

ax.set(
    title="Country-block Prediction Strength — higher is better",
    xlabel="Number of clusters, k",
    ylabel="Prediction Strength",
    xticks=list(CANDIDATE_K),
    ylim=(0, 1.02),
)
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

if not selected_k_values:
    display(Markdown(
        f"""
#### Objective selection result

No candidate `k = 2,...,10` reaches the working Prediction Strength threshold
of **{PS_THRESHOLD:.2f}**. Under the stated rule, **no value of `k` is selected**.
The other diagnostics may still be inspected, but they do not override this rule.
"""
    ))
else:
    selected_ps_text = ", ".join(
        f"`k = {k}` ({prediction_strength_summary.loc[k, 'prediction_strength_mean']:.3f})"
        for k in selected_k_values
    )

    best_silhouette_k = int(baseline_metrics["silhouette"].idxmax())
    best_silhouette = baseline_metrics.loc[
        best_silhouette_k, "silhouette"
    ]

    display(Markdown(
        f"""
#### Objective selection result

The candidates meeting the working Prediction Strength threshold are:

**{selected_ps_text}**

All of these values are retained as **supported development-only resolutions**
under the fixed working mean-PS rule. The finest supported resolution is
**`k = {finest_selected_k}`**.

The table and plot should still be read carefully: individual evaluations, SD,\nminimum, maximum, and the percentage of evaluations above threshold show how\nconsistent the solution is across country halves. These quantities are diagnostic;\n**only the mean Prediction Strength is used by the Week 3 retention rule**.

For comparison, silhouette is highest at **`k = {best_silhouette_k}`**
({best_silhouette:.3f}). The two measures answer different questions:
silhouette identifies the clearest geometric separation, whereas Prediction Strength
tests whether a clustering resolution remains reproducible across development-country
splits. The threshold therefore defines a **set of supported resolutions**, not a
single compulsory winner.
"""
    ))


### 11.6 Check the prevalence of all selected cluster solutions


This section displays cluster prevalence **only for the values of `k` retained by the mean Prediction Strength rule**.

For each retained solution, the table reports:

- ordinary respondent percentage;
- equal-country-weighted percentage;
- respondent percentage minus equal-country-weighted percentage.

It also repeats the solution's cluster-size imbalance and largest absolute prevalence gap.

These are descriptive properties of the already fitted weighted K-means solutions. They do not enter the Prediction Strength selection rule, and the respondent/equal-country difference does not represent a comparison with an unweighted K-means refit.


In [ ]:
if not selected_k_values:
    display(Markdown(
        "No cluster-prevalence tables are shown because no candidate passed "
        "the Prediction Strength threshold."
    ))
else:
    for selected_k in selected_k_values:
        selected_cluster_sizes = (
            baseline_cluster_sizes.loc[
                baseline_cluster_sizes["k"].eq(selected_k),
                [
                    "cluster",
                    "respondent_pct",
                    "equal_country_weighted_pct",
                ],
            ]
            .copy()
            .sort_values("cluster")
        )

        selected_cluster_sizes["weighting_gap_pp"] = (
            selected_cluster_sizes["respondent_pct"]
            - selected_cluster_sizes["equal_country_weighted_pct"]
        )

        selected_sizes_display = selected_cluster_sizes.copy().round(2)
        selected_sizes_display.columns = pd.MultiIndex.from_tuples([
            ("Cluster", ""),
            ("Respondents", "%"),
            ("Equal-country", "%"),
            ("Difference", "pp"),
        ])

        display(Markdown(f"#### Selected solution: k = {selected_k}"))
        display(selected_sizes_display)

        selected_imbalance = baseline_metrics.loc[
            selected_k, "cluster_size_imbalance_pct"
        ]
        selected_weighting_gap = baseline_metrics.loc[
            selected_k, "max_weighting_share_gap_pct_points"
        ]

        display(Markdown(
            f"""
Cluster-size imbalance: **{selected_imbalance:.1f}%**. Largest respondent-versus-
equal-country prevalence difference: **{selected_weighting_gap:.2f} percentage points**.
"""
        ))


### 11.7 Ask whether simple burden scores explain the selected cluster labels


**What this cell does:** for every Prediction-Strength-retained value of `k`, constructs two respondent-level burden scores from the **original non-imputed profile variables**:

- **overall worry:** the mean of the observed six worry variables;
- **overall harm exposure:** the mean of the observed seven binary harm indicators, i.e. the fraction of observed domains reporting harm to the respondent or someone personally known.

It then uses five-fold `GroupKFold` by country to train shallow decision trees that predict the fixed full-development K-means labels from only those two scores. Any missing burden score is median-imputed within the training countries for that fold. Trees of depth 3 and 5 are reported, with `min_samples_leaf = 200`. Tree fitting uses the equal-country respondent weights.

The output summarizes equal-country-weighted accuracy, balanced accuracy, macro-F1, and their country-fold variation. `1/k` is shown as the balanced-accuracy value of a constant single-cluster guess.

**Why it matters:** this is an **explanatory-fidelity test**, not independent clustering validation. If two aggregate burden scores reproduce the labels very well, the partition may mainly reflect overall low/medium/high burden rather than distinct multidomain compositions.

The target labels were produced by the full-development K-means fit, so Prediction Strength—not this tree analysis—remains the geographic reproducibility test.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier

BURDEN_TREE_DEPTHS = [3, 5]
BURDEN_TREE_MIN_LEAF = 200
BURDEN_TREE_FOLDS = 5

worry_profile_features = list(WORRY_FEATURES.values())
harm_profile_features = list(EXPERIENCE_FEATURES.values())

burden_scores = pd.DataFrame({
    "overall_worry": development_data[worry_profile_features].mean(axis=1),
    "overall_harm": development_data[harm_profile_features].mean(axis=1),
})

country_groups = development_data["COUNTRY_ISO3"].to_numpy()
burden_cv = GroupKFold(n_splits=BURDEN_TREE_FOLDS)

burden_fidelity_rows = []

for selected_k in selected_k_values:
    target_labels = baseline_models[selected_k]["labels"]

    for depth in BURDEN_TREE_DEPTHS:
        for fold, (train_indices, test_indices) in enumerate(
            burden_cv.split(burden_scores, groups=country_groups),
            start=1,
        ):
            fold_imputer = SimpleImputer(strategy="median")

            train_x = fold_imputer.fit_transform(
                burden_scores.iloc[train_indices]
            )
            test_x = fold_imputer.transform(
                burden_scores.iloc[test_indices]
            )

            tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_leaf=BURDEN_TREE_MIN_LEAF,
                random_state=KMEANS_SEED,
            )
            tree.fit(
                train_x,
                target_labels[train_indices],
                sample_weight=analysis_weights[train_indices],
            )

            predicted = tree.predict(test_x)

            burden_fidelity_rows.append({
                "k": selected_k,
                "depth": depth,
                "fold": fold,
                "weighted_accuracy": np.average(
                    predicted == target_labels[test_indices],
                    weights=analysis_weights[test_indices],
                ),
                "balanced_accuracy": balanced_accuracy_score(
                    target_labels[test_indices],
                    predicted,
                    sample_weight=analysis_weights[test_indices],
                ),
                "macro_f1": f1_score(
                    target_labels[test_indices],
                    predicted,
                    average="macro",
                    sample_weight=analysis_weights[test_indices],
                    zero_division=0,
                ),
                "constant_guess_balanced_accuracy": 1 / selected_k,
            })

burden_fidelity_folds = pd.DataFrame(burden_fidelity_rows)

if burden_fidelity_folds.empty:
    burden_fidelity_summary = pd.DataFrame()
    display(Markdown(
        "No burden-score fidelity analysis is produced because no candidate "
        "passed the Prediction Strength threshold."
    ))
else:
    burden_fidelity_summary = (
        burden_fidelity_folds
        .groupby(["k", "depth"])
        .agg(
            weighted_accuracy_mean=("weighted_accuracy", "mean"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd=("balanced_accuracy", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            constant_guess_balanced_accuracy=(
                "constant_guess_balanced_accuracy",
                "first",
            ),
            n_country_folds=("fold", "count"),
        )
    )

    burden_fidelity_display = burden_fidelity_summary.copy().round(3)
    burden_fidelity_display.columns = pd.MultiIndex.from_tuples([
        ("Weighted accuracy", "mean"),
        ("Balanced accuracy", "mean"),
        ("Balanced accuracy", "SD"),
        ("Macro-F1", "mean"),
        ("Constant guess", "balanced acc."),
        ("Country folds", "N"),
    ])

    display(burden_fidelity_display)

    display(Markdown(
        """
Read this table as **explanatory fidelity**, not as a competition between
classifiers. Values close to 1 mean that the selected cluster labels can be
reconstructed well from only overall worry and overall harm. Lower values mean
that these two aggregate scores are insufficient to reproduce the partition with
the deliberately simple trees used here.
"""
    ))


### 11.8 Check whether one survey feature almost defines a selected cluster


**What this cell does:** for every retained `k` and every cluster, turns membership in that cluster into a one-vs-rest target and asks how well a **single original profile feature plus one threshold** can reconstruct membership.

Five country-block folds are used. In each fold:

1. only the training countries are searched for the feature, threshold, and direction with the highest weighted balanced accuracy;
2. that one selected rule is applied unchanged to the validation countries;
3. validation precision, recall, weighted balanced accuracy, and observed-weight coverage are recorded.

The search considers only thresholds between the actually observed values of each feature. Missing values in the chosen feature are **excluded from that fold's rule evaluation rather than imputed**, and the coverage column reports how much validation analysis weight remained observed.

The summary shows the feature selected most often across the five folds and the number of folds selecting it. Its performance columns, however, summarize the **fold-specific selected rules**, not one single full-development rule.

**Why it matters:** a geographically reproducible cluster may still be scientifically trivial if membership is almost equivalent to one answer such as `Water harm = 1`. This diagnostic complements the burden-score check without imposing an arbitrary cutoff for “too simple.”

Like the burden analysis, this explains the fixed development labels; it does not independently validate the clustering and does not remove or refit any feature.


In [ ]:
ONE_FEATURE_FOLDS = 5

one_feature_readable_map = {
    "worry_food": "Food worry",
    "worry_water": "Water worry",
    "worry_crime": "Crime worry",
    "worry_severe_weather": "Severe-weather worry",
    "worry_traffic": "Traffic worry",
    "worry_mental_health": "Mental-health worry",
    "harm_food": "Food harm",
    "harm_water": "Water harm",
    "harm_crime": "Crime harm",
    "harm_severe_weather": "Severe-weather harm",
    "harm_traffic": "Traffic harm",
    "harm_mental_health": "Mental-health harm",
    "harm_work": "Work harm",
}

assert set(PROFILE_FEATURES) == set(one_feature_readable_map)


def find_best_single_feature_rule(frame, target, weights):
    """Choose the one-feature threshold rule with highest weighted balanced accuracy."""
    best_rule = None

    for feature in frame.columns:
        values = frame[feature].to_numpy(dtype=float)
        observed = np.isfinite(values)

        if observed.sum() == 0:
            continue

        observed_values = np.unique(values[observed])
        if len(observed_values) < 2:
            continue

        observed_target = target[observed]
        if np.unique(observed_target).size < 2:
            continue

        thresholds = (observed_values[:-1] + observed_values[1:]) / 2

        for threshold in thresholds:
            for above in [True, False]:
                predicted = (
                    values[observed] > threshold
                    if above
                    else values[observed] <= threshold
                )

                score = balanced_accuracy_score(
                    observed_target,
                    predicted,
                    sample_weight=weights[observed],
                )

                candidate = {
                    "balanced_accuracy": score,
                    "feature": feature,
                    "threshold": float(threshold),
                    "above": above,
                }

                if (
                    best_rule is None
                    or candidate["balanced_accuracy"]
                    > best_rule["balanced_accuracy"]
                ):
                    best_rule = candidate

    return best_rule


one_feature_frame = development_data[PROFILE_FEATURES]
one_feature_cv = GroupKFold(n_splits=ONE_FEATURE_FOLDS)
one_feature_rows = []

for selected_k in selected_k_values:
    cluster_labels = baseline_models[selected_k]["labels"]

    for cluster in range(selected_k):
        binary_target = cluster_labels == cluster

        for fold, (train_indices, test_indices) in enumerate(
            one_feature_cv.split(
                one_feature_frame,
                groups=country_groups,
            ),
            start=1,
        ):
            rule = find_best_single_feature_rule(
                one_feature_frame.iloc[train_indices],
                binary_target[train_indices],
                analysis_weights[train_indices],
            )

            if rule is None:
                continue

            test_values = (
                one_feature_frame.iloc[test_indices][rule["feature"]]
                .to_numpy(dtype=float)
            )

            observed = np.isfinite(test_values)
            target = binary_target[test_indices][observed]
            weights = analysis_weights[test_indices][observed]

            if rule["above"]:
                predicted = test_values[observed] > rule["threshold"]
                operator = ">"
            else:
                predicted = test_values[observed] <= rule["threshold"]
                operator = "<="

            true_positive_weight = weights[target & predicted].sum()
            predicted_positive_weight = weights[predicted].sum()
            target_positive_weight = weights[target].sum()

            precision = (
                true_positive_weight / predicted_positive_weight
                if predicted_positive_weight > 0
                else np.nan
            )
            recall = (
                true_positive_weight / target_positive_weight
                if target_positive_weight > 0
                else np.nan
            )

            one_feature_rows.append({
                "k": selected_k,
                "cluster": cluster,
                "fold": fold,
                "feature": rule["feature"],
                "feature_name": one_feature_readable_map[rule["feature"]],
                "threshold": rule["threshold"],
                "operator": operator,
                "precision": precision,
                "recall": recall,
                "balanced_accuracy": balanced_accuracy_score(
                    target,
                    predicted,
                    sample_weight=weights,
                ),
                "observed_weight_pct": (
                    100
                    * weights.sum()
                    / analysis_weights[test_indices].sum()
                ),
            })

one_feature_folds = pd.DataFrame(one_feature_rows)

if one_feature_folds.empty:
    one_feature_summary = pd.DataFrame()

    display(Markdown(
        "No one-feature dominance analysis is produced because no candidate "
        "passed the Prediction Strength threshold."
    ))

else:
    performance_summary = (
        one_feature_folds
        .groupby(["k", "cluster"])
        .agg(
            precision_mean=("precision", "mean"),
            recall_mean=("recall", "mean"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd=("balanced_accuracy", "std"),
            observed_weight_pct_mean=("observed_weight_pct", "mean"),
            n_country_folds=("fold", "count"),
        )
    )

    feature_frequency = (
        one_feature_folds
        .groupby(["k", "cluster", "feature_name"])
        .size()
        .rename("selected_folds")
        .reset_index()
        .sort_values(
            ["k", "cluster", "selected_folds", "feature_name"],
            ascending=[True, True, False, True],
        )
        .drop_duplicates(["k", "cluster"])
        .set_index(["k", "cluster"])
    )

    one_feature_summary = (
        performance_summary
        .join(feature_frequency[["feature_name", "selected_folds"]])
        .reset_index()
    )

    one_feature_display = one_feature_summary[
        [
            "k",
            "cluster",
            "feature_name",
            "selected_folds",
            "precision_mean",
            "recall_mean",
            "balanced_accuracy_mean",
            "balanced_accuracy_sd",
            "observed_weight_pct_mean",
        ]
    ].copy().round(3)

    one_feature_display.columns = pd.MultiIndex.from_tuples([
        ("k", ""),
        ("Cluster", ""),
        ("Most selected feature", ""),
        ("Selected", "folds"),
        ("Precision", "mean"),
        ("Recall", "mean"),
        ("Balanced accuracy", "mean"),
        ("Balanced accuracy", "SD"),
        ("Observed weight", "%"),
    ])

    display(one_feature_display)

    display(Markdown(
        """
Higher precision and recall together indicate that a single survey item closely
matches the cluster. Balanced accuracy near `0.5` is little better than an
uninformative binary rule; values approaching `1` indicate increasingly strong
one-feature reconstruction. The feature name column shows which item was selected
most often across the five training folds, while the performance values summarize
the fold-specific selected rules.
"""
    ))

### 11.9 Compare and narrate the selected clusters in the original response units


**What this cell does:** first plots inverse-transformed K-means centroids for **all** candidate values of `k`. It then describes only the Prediction-Strength-retained solutions using the original, non-imputed survey responses.

For each retained cluster and each of the 13 features, three weighted response distributions are calculated among respondents with that feature observed:

- the cluster itself;
- all profile-eligible development respondents;
- all profile-eligible development respondents outside that cluster.

The comparison metric is **total-variation (TV) distance**, reported as a 0–100 **distributional contrast**. For binary harm variables, this equals the absolute percentage-point difference in the observed harm prevalence.

### Selecting features for the narrative

For each cluster, the 13 features are ranked by **cluster-vs-outside distributional contrast**. The code finds the largest adjacent drop in that ordered list and selects the features above that break. Thus the number of highlighted features can differ by cluster; no fixed top-three or top-four rule is imposed. If all contrasts are effectively identical, no compact subset is forced.

The matrices still display all 13 features. Their symbols mean:

- `↑` = higher worry or more reported harm than the reference;
- `↓` = lower worry or less reported harm;
- `↔` = a mixed shift in the three-level worry distribution;
- `*` = selected by the largest-break rule for that cluster.

The narrative itself returns to the original response units rather than interpreting TV percentages directly:

- worry descriptions compare the percentages at `2 = very worried` and `0 = not worried`;
- harm descriptions compare the observed percentage reporting respondent-or-known-person harm.

Observed-weight coverage is retained in the profile data so missing responses are not silently treated as substantive categories.

**Why it matters:** this section gives the candidate clusters a factual semantic description. It does not turn descriptive contrasts into additional validation evidence. The preceding burden and one-feature checks remain important when deciding whether a seemingly distinctive profile is actually multidomain.


In [ ]:
readable_feature_names = [
    "Food worry", "Water worry", "Crime worry", "Weather worry",
    "Traffic worry", "Mental-health worry", "Food harm", "Water harm",
    "Crime harm", "Weather harm", "Traffic harm", "Mental-health harm",
    "Work harm",
]
readable_feature_map = dict(zip(PROFILE_FEATURES, readable_feature_names))

# A. Keep the centroid view for all candidate solutions.
centroid_tables = []
fig, axes = plt.subplots(3, 3, figsize=(20, 17))

for axis, k in zip(axes.flat, CANDIDATE_K):
    model = baseline_models[k]["model"]
    original_centroids = profile_scaler.inverse_transform(model.cluster_centers_)

    centroid_table = pd.DataFrame(original_centroids, columns=PROFILE_FEATURES)
    centroid_table.insert(0, "cluster", range(k))
    centroid_table.insert(0, "k", k)
    centroid_tables.append(centroid_table)

    sns.heatmap(
        pd.DataFrame(original_centroids, columns=readable_feature_names),
        vmin=0,
        vmax=2,
        cmap="YlOrRd",
        cbar=(k == max(CANDIDATE_K)),
        ax=axis,
    )
    axis.set(title=f"k = {k}", xlabel="", ylabel="Cluster")
    axis.tick_params(axis="x", rotation=65, labelsize=8)

plt.suptitle(
    "Development-only K-means centroids in original feature units",
    y=1.01,
    fontsize=16,
)
plt.tight_layout()
plt.show()

baseline_centroids = pd.concat(centroid_tables, ignore_index=True)


# B. Describe every Prediction-Strength-selected solution using original,
#    non-imputed response distributions and two reference populations.
worry_features = set(WORRY_FEATURES.values())
harm_features = set(EXPERIENCE_FEATURES.values())


def weighted_distribution(values, weights, categories):
    observed = values.notna()
    observed_values = values.loc[observed]
    observed_weights = weights.loc[observed]

    total_observed_weight = observed_weights.sum()
    total_weight = weights.sum()

    if total_observed_weight == 0:
        distribution = {category: np.nan for category in categories}
        observed_weight_pct = 0.0
    else:
        distribution = {
            category: (
                observed_weights.loc[observed_values.eq(category)].sum()
                / total_observed_weight
            )
            for category in categories
        }
        observed_weight_pct = 100 * total_observed_weight / total_weight

    return distribution, observed_weight_pct


def distributional_contrast_pct(distribution_a, distribution_b, categories):
    """Total-variation distance expressed on a 0--100 percentage scale."""
    return 50 * sum(
        abs(distribution_a[category] - distribution_b[category])
        for category in categories
    )


def worry_direction(cluster_distribution, reference_distribution):
    high_delta = cluster_distribution[2] - reference_distribution[2]
    low_delta = cluster_distribution[0] - reference_distribution[0]

    if high_delta > 0 and low_delta < 0:
        return "toward higher worry"
    if high_delta < 0 and low_delta > 0:
        return "toward lower worry"
    return "mixed pattern"


def format_worry_distribution(distribution):
    return " / ".join(
        f"{category}: {100 * distribution[category]:.1f}%"
        for category in [0, 1, 2]
    )


def format_harm_distribution(distribution):
    return f"{100 * distribution[1]:.1f}% harm"


distribution_rows = []

for selected_k in selected_k_values:
    labels = baseline_models[selected_k]["labels"]

    for cluster in range(selected_k):
        cluster_mask = labels == cluster
        outside_mask = ~cluster_mask

        cluster_weights = development_data.loc[
            cluster_mask, "weight_equal_country"
        ]
        outside_weights = development_data.loc[
            outside_mask, "weight_equal_country"
        ]

        for feature in PROFILE_FEATURES:
            if feature in worry_features:
                categories = [0, 1, 2]
                feature_type = "worry"
            else:
                categories = [0, 1]
                feature_type = "harm"

            development_distribution, development_observed_weight_pct = weighted_distribution(
                development_data[feature],
                development_data["weight_equal_country"],
                categories,
            )
            cluster_distribution, cluster_observed_weight_pct = weighted_distribution(
                development_data.loc[cluster_mask, feature],
                cluster_weights,
                categories,
            )
            outside_distribution, outside_observed_weight_pct = weighted_distribution(
                development_data.loc[outside_mask, feature],
                outside_weights,
                categories,
            )

            contrast_vs_development_pct = distributional_contrast_pct(
                cluster_distribution,
                development_distribution,
                categories,
            )
            contrast_vs_outside_pct = distributional_contrast_pct(
                cluster_distribution,
                outside_distribution,
                categories,
            )

            if feature_type == "worry":
                direction_vs_development = worry_direction(
                    cluster_distribution, development_distribution
                )
                direction_vs_outside = worry_direction(
                    cluster_distribution, outside_distribution
                )
                harm_difference_vs_development_pp = np.nan
                harm_difference_vs_outside_pp = np.nan
            else:
                harm_difference_vs_development_pp = 100 * (
                    cluster_distribution[1] - development_distribution[1]
                )
                harm_difference_vs_outside_pp = 100 * (
                    cluster_distribution[1] - outside_distribution[1]
                )

                direction_vs_development = (
                    "more reported harm" if harm_difference_vs_development_pp > 0
                    else "less reported harm" if harm_difference_vs_development_pp < 0
                    else "same reported harm"
                )
                direction_vs_outside = (
                    "more reported harm" if harm_difference_vs_outside_pp > 0
                    else "less reported harm" if harm_difference_vs_outside_pp < 0
                    else "same reported harm"
                )

            row = {
                "k": selected_k,
                "cluster": cluster,
                "feature": feature,
                "feature_name": readable_feature_map[feature],
                "feature_type": feature_type,
                "contrast_vs_development_pct": contrast_vs_development_pct,
                "contrast_vs_outside_pct": contrast_vs_outside_pct,
                "direction_vs_development": direction_vs_development,
                "direction_vs_outside": direction_vs_outside,
                "cluster_distribution": (
                    format_worry_distribution(cluster_distribution)
                    if feature_type == "worry"
                    else format_harm_distribution(cluster_distribution)
                ),
                "development_distribution": (
                    format_worry_distribution(development_distribution)
                    if feature_type == "worry"
                    else format_harm_distribution(development_distribution)
                ),
                "outside_distribution": (
                    format_worry_distribution(outside_distribution)
                    if feature_type == "worry"
                    else format_harm_distribution(outside_distribution)
                ),
                "harm_difference_vs_development_pp": harm_difference_vs_development_pp,
                "harm_difference_vs_outside_pp": harm_difference_vs_outside_pp,
                "cluster_observed_weight_pct": cluster_observed_weight_pct,
                "development_observed_weight_pct": development_observed_weight_pct,
                "outside_observed_weight_pct": outside_observed_weight_pct,
            }

            for category in [0, 1, 2]:
                row[f"cluster_category_{category}_pct"] = (
                    100 * cluster_distribution.get(category, np.nan)
                )
                row[f"development_category_{category}_pct"] = (
                    100 * development_distribution.get(category, np.nan)
                )
                row[f"outside_category_{category}_pct"] = (
                    100 * outside_distribution.get(category, np.nan)
                )

            distribution_rows.append(row)

baseline_distribution_profiles = pd.DataFrame(distribution_rows)


def select_features_at_largest_break(cluster_profile):
    """
    Rank features by cluster-vs-outside contrast and cut at the largest
    adjacent drop. No fixed number of features is imposed.
    """
    ranked = (
        cluster_profile
        .sort_values("contrast_vs_outside_pct", ascending=False)
        .reset_index(drop=True)
        .copy()
    )

    contrasts = ranked["contrast_vs_outside_pct"].to_numpy(dtype=float)

    if len(contrasts) == 0:
        return ranked.iloc[0:0], {
            "n_informative": 0,
            "break_after_rank": np.nan,
            "largest_drop_pct_points": np.nan,
        }

    if len(contrasts) == 1 or np.allclose(contrasts, contrasts[0]):
        return ranked.iloc[0:0], {
            "n_informative": 0,
            "break_after_rank": np.nan,
            "largest_drop_pct_points": 0.0,
        }

    successive_drops = contrasts[:-1] - contrasts[1:]
    break_index = int(np.argmax(successive_drops))
    n_informative = break_index + 1
    selected = ranked.iloc[:n_informative].copy()

    return selected, {
        "n_informative": n_informative,
        "break_after_rank": n_informative,
        "largest_drop_pct_points": float(successive_drops[break_index]),
    }


informative_feature_rows = []
informative_selection_rows = []

for selected_k in selected_k_values:
    for cluster in range(selected_k):
        cluster_profile = baseline_distribution_profiles.loc[
            baseline_distribution_profiles["k"].eq(selected_k)
            & baseline_distribution_profiles["cluster"].eq(cluster)
        ].copy()

        selected_rows, audit = select_features_at_largest_break(cluster_profile)
        informative_feature_rows.append(selected_rows)

        informative_selection_rows.append({
            "k": selected_k,
            "cluster": cluster,
            "n_informative": audit["n_informative"],
            "break_after_rank": audit["break_after_rank"],
            "largest_drop_pct_points": audit["largest_drop_pct_points"],
            "informative_features": ", ".join(selected_rows["feature_name"].tolist()),
        })

informative_features = (
    pd.concat(informative_feature_rows, ignore_index=True)
    if informative_feature_rows
    else pd.DataFrame()
)
informative_feature_selection = pd.DataFrame(informative_selection_rows)

informative_feature_keys = set(
    zip(
        informative_features.get("k", pd.Series(dtype=int)),
        informative_features.get("cluster", pd.Series(dtype=int)),
        informative_features.get("feature", pd.Series(dtype=str)),
    )
)


def direction_symbol(row, reference):
    if row["feature_type"] == "worry":
        direction = row[f"direction_vs_{reference}"]
        if direction == "toward higher worry":
            return "↑"
        if direction == "toward lower worry":
            return "↓"
        return "↔"

    difference = row[f"harm_difference_vs_{reference}_pp"]
    if difference > 0:
        return "↑"
    if difference < 0:
        return "↓"
    return "·"


def matrix_cell(row, reference):
    symbol = direction_symbol(row, reference)
    contrast = row[f"contrast_vs_{reference}_pct"]
    star = " *" if (
        row["k"], row["cluster"], row["feature"]
    ) in informative_feature_keys else ""
    return f"{symbol} {contrast:.1f}%{star}"


feature_order = list(WORRY_FEATURES.values()) + list(EXPERIENCE_FEATURES.values())

feature_index = pd.MultiIndex.from_tuples(
    [
        (
            "Worry" if feature in worry_features else "Harm",
            readable_feature_map[feature],
        )
        for feature in feature_order
    ],
    names=["Type", "Feature"],
)


if not selected_k_values:
    display(Markdown(
        "No distributional comparison matrices or narratives are produced because "
        "no candidate passed the Prediction Strength threshold."
    ))
else:
    for selected_k in selected_k_values:
        selected_profiles = baseline_distribution_profiles.loc[
            baseline_distribution_profiles["k"].eq(selected_k)
        ].copy()

        display(Markdown(
            f"#### Feature-by-cluster comparison for selected k = {selected_k}"
        ))

        development_matrix = pd.DataFrame(
            index=feature_index,
            columns=[f"Cluster {cluster}" for cluster in range(selected_k)],
            dtype=object,
        )
        outside_matrix = pd.DataFrame(
            index=feature_index,
            columns=[f"Cluster {cluster}" for cluster in range(selected_k)],
            dtype=object,
        )

        for _, row in selected_profiles.iterrows():
            row_key = (
                "Worry" if row["feature_type"] == "worry" else "Harm",
                row["feature_name"],
            )
            cluster_column = f"Cluster {int(row['cluster'])}"

            development_matrix.loc[row_key, cluster_column] = matrix_cell(
                row, "development"
            )
            outside_matrix.loc[row_key, cluster_column] = matrix_cell(
                row, "outside"
            )

        display(Markdown(
            "**Vs development overall** — same fixed reference for every cluster"
        ))
        display(development_matrix)

        display(Markdown(
            "**Vs everybody outside the cluster** — used to identify the "
            "informative feature subset"
        ))
        display(outside_matrix)

        display(Markdown(
            "`↑` higher worry / more harm; `↓` lower worry / less harm; "
            "`↔` mixed worry pattern; percentage = TV distributional contrast; "
            "`*` feature above the cluster-specific natural break."
        ))

        selection_display = (
            informative_feature_selection.loc[
                informative_feature_selection["k"].eq(selected_k),
                [
                    "cluster",
                    "n_informative",
                    "break_after_rank",
                    "largest_drop_pct_points",
                    "informative_features",
                ],
            ]
            .copy()
            .round(2)
        )
        selection_display.columns = pd.MultiIndex.from_tuples([
            ("Cluster", ""),
            ("Informative", "N"),
            ("Break after", "rank"),
            ("Largest drop", "pp"),
            ("Selected features", ""),
        ])

        display(Markdown("##### Natural-break feature selection"))
        display(selection_display)

        display(Markdown("##### Factual cluster narratives"))

        for cluster in range(selected_k):
            cluster_informative = informative_features.loc[
                informative_features["k"].eq(selected_k)
                & informative_features["cluster"].eq(cluster)
            ].sort_values("contrast_vs_outside_pct", ascending=False)

            if cluster_informative.empty:
                display(Markdown(
                    f"**Cluster {cluster}**  \n"
                    "No compact informative feature subset is evident because the "
                    "cluster-vs-outside contrasts do not show a distinct break."
                ))
                continue

            narrative_lines = [f"**Cluster {cluster}**", ""]

            for _, row in cluster_informative.iterrows():
                if row["feature_type"] == "worry":
                    if row["direction_vs_outside"] == "toward higher worry":
                        descriptor = "higher worry"
                    elif row["direction_vs_outside"] == "toward lower worry":
                        descriptor = "lower worry"
                    else:
                        descriptor = "different worry pattern"

                    narrative_lines.append(
                        f"- **{row['feature_name']} — {descriptor}:** "
                        f"category 2 is {row['cluster_category_2_pct']:.1f}% in the cluster, "
                        f"{row['outside_category_2_pct']:.1f}% outside, and "
                        f"{row['development_category_2_pct']:.1f}% in development overall; "
                        f"category 0 is {row['cluster_category_0_pct']:.1f}%, "
                        f"{row['outside_category_0_pct']:.1f}%, and "
                        f"{row['development_category_0_pct']:.1f}%, respectively."
                    )
                else:
                    difference = row["harm_difference_vs_outside_pp"]
                    sign = "+" if difference > 0 else ""
                    narrative_lines.append(
                        f"- **{row['feature_name']}:** "
                        f"{row['cluster_category_1_pct']:.1f}% report harm in the cluster "
                        f"versus {row['outside_category_1_pct']:.1f}% outside "
                        f"({sign}{difference:.1f} percentage points); "
                        f"development overall is {row['development_category_1_pct']:.1f}%."
                    )

            display(Markdown("\n".join(narrative_lines)))


### 11.10 Save the development-only selection and interpretation evidence


**What this cell does:** saves the Week 3 development-only evidence:

- baseline K-means diagnostics and complete all-`k` cluster-size tables;
- centroids;
- Prediction Strength results for the 20 repeated country-half evaluations per `k` and their summaries;
- burden-score explanatory-fidelity results;
- one-feature explanatory-fidelity results;
- original-unit distributional profiles;
- natural-break informative-feature selections;
- the selected `k` values and the status of the `0.75` working threshold.

**Why it matters:** the saved artifacts separate reproducibility, simplicity checks, and semantic description rather than collapsing them into one composite score.

No clustering solution is refitted after the burden or one-feature explanations. Feature ablation, alternative geometries, alternative clustering algorithms, and other robustness analyses remain Week 4 tasks.

This section uses only the development dataframe already loaded in Section 11. The held-out-country respondent file is not opened or used anywhere in the modelling section.


In [ ]:
baseline_metrics.reset_index().to_csv(
    ARTIFACTS / "week03_baseline_kmeans_metrics.csv", index=False
)
baseline_cluster_sizes.to_csv(
    ARTIFACTS / "week03_baseline_kmeans_cluster_sizes.csv", index=False
)
baseline_centroids.to_csv(
    ARTIFACTS / "week03_baseline_kmeans_centroids.csv", index=False
)
prediction_strength_folds.to_csv(
    ARTIFACTS / "week03_prediction_strength_folds.csv", index=False
)
prediction_strength_summary.reset_index().to_csv(
    ARTIFACTS / "week03_prediction_strength_summary.csv", index=False
)


if not burden_fidelity_folds.empty:
    burden_fidelity_folds.to_csv(
        ARTIFACTS / "week03_burden_fidelity_folds.csv", index=False
    )

if not burden_fidelity_summary.empty:
    burden_fidelity_summary.reset_index().to_csv(
        ARTIFACTS / "week03_burden_fidelity_summary.csv", index=False
    )


if not one_feature_folds.empty:
    one_feature_folds.to_csv(
        ARTIFACTS / "week03_one_feature_folds.csv", index=False
    )

if not one_feature_summary.empty:
    one_feature_summary.to_csv(
        ARTIFACTS / "week03_one_feature_summary.csv", index=False
    )

if not baseline_distribution_profiles.empty:
    baseline_distribution_profiles.to_csv(
        ARTIFACTS / "week03_selected_k_distribution_profiles.csv", index=False
    )

if not informative_features.empty:
    informative_features.to_csv(
        ARTIFACTS / "week03_selected_k_informative_features.csv", index=False
    )

if not informative_feature_selection.empty:
    informative_feature_selection.to_csv(
        ARTIFACTS / "week03_selected_k_feature_selection.csv", index=False
    )

selection_record = pd.Series({
    "prediction_strength_threshold": PS_THRESHOLD,
    "prediction_strength_repeats": PS_REPEATS,
    "prediction_strength_evaluations_per_k": 2 * PS_REPEATS,
    "selected_k_values": selected_k_values,
    "finest_selected_k": finest_selected_k,
    "selection_rule": "all k with mean country-block Prediction Strength >= threshold",
    "threshold_status": (
        "chosen during exploratory development and fixed for subsequent comparisons"
    ),
})
selection_record.to_json(
    ARTIFACTS / "week03_k_selection.json",
    indent=2,
)

print("Saved development-only K-means, Prediction Strength, burden-fidelity, one-feature, and profile evidence.")
if not selected_k_values:
    print("No candidate k passed the working Prediction Strength threshold.")
else:
    print(
        "Selected development-only resolutions: "
        + ", ".join(f"k = {k}" for k in selected_k_values)
        + "."
    )
    print(f"Finest supported resolution: k = {finest_selected_k}.")
    print("Saved factual original-unit narratives and natural-break feature selections.")
print("No held-out-country respondent data were used.")


## 12. Collect transparent numerical checkpoints


**What this cell does:** collects a compact set of scalar checkpoints from the completed Week 3 run: total source rows, feature counts, fully unobserved development rows excluded from clustering, country counts, candidate `k` range, Prediction Strength threshold, and retained resolutions. The number of Prediction Strength evaluations is determined from `PS_REPEATS`, so the saved selection metadata now records 20 evaluations per `k`.

**Why it matters:** these values provide reproducible checks that students ran the intended analysis and can support automatically graded quiz questions. The held-out-country count is retained as a scalar from the preparation stage before respondent-level preparation objects are removed; Section 11 does not reload the held-out-country data.


In [ ]:
week03_checkpoints = pd.Series({
    "analysis rows": TOTAL_RESPONDENTS,
    "profile features": len(PROFILE_FEATURES),
    "fully missing development profiles excluded": EXCLUDED_FULLY_MISSING_PROFILES,
    "ordered worry features": len(WORRY_FEATURES),
    "binary experience features": len(EXPERIENCE_FEATURES),
    "development countries": development_data["COUNTRY_ISO3"].nunique(),
    "held-out countries": HELDOUT_COUNTRIES,
    "candidate k values examined": len(list(CANDIDATE_K)),
    "largest candidate k": max(CANDIDATE_K),
    "prediction strength threshold": PS_THRESHOLD,
    "selected k values": ", ".join(map(str, selected_k_values)) if selected_k_values else "none",
    "finest selected k": finest_selected_k if finest_selected_k is not None else "none",
}, name="answer")
display(week03_checkpoints.to_frame())


## Week 3 conclusion


The Week 3 notebook maintains a clear separation between **data curation**, **development-only model selection**, and the **external held-out countries**.

Raw survey codes are validated before recoding. The six worry variables are represented as ordered `0,1,2` features, while the seven binary harm variables represent reported harm to the respondent or someone personally known. The exact country split is generated once and then reused from the frozen registry, with source and split hashes recorded for reproducibility.

Missingness is audited only in the development countries. Incomplete respondents remain in the frozen data; only respondents with no observed value across all 13 profile features are excluded from clustering. Remaining missing values are median-imputed for the Week 3 baseline.

The clustering baseline uses unweighted z-standardization followed by equal-country-weighted K-means for `k = 2,...,10`. K-means is fitted on the full profile-eligible development dataset. A fixed sample of up to 10,000 development respondents is used only for the unweighted silhouette, Calinski--Harabasz, and Davies--Bouldin diagnostics. Weighted inertia, cluster sizes, and initialization stability are based on the full development fit or full-development refits. Explained weighted dispersion is still exported but is no longer emphasized because it is a normalized restatement of inertia.

Country-block Prediction Strength is the primary Week 3 reproducibility rule. The analysis deliberately keeps a demanding **repeated two-half country design** rather than ordinary 5-fold cross-validation: each transported model is trained on only about half of the development countries. The number of repeats is increased to 10, and both directions are evaluated, producing **20 evaluations per `k`**. Imputation and scaling are refitted inside every training-country half.

The Prediction Strength plot now shows all individual country-half evaluations together with the mean and ±1 SD. The SD bar describes variability; it is not a confidence interval and does not alter the selection rule. Every `k` with **mean** Prediction Strength at least the fixed working threshold `0.75` is retained. A solution is therefore not retained merely because some individual evaluations—or its mean-plus-SD—exceed `0.75`.

For the retained resolutions, two explanatory checks ask whether the K-means labels are largely reducible to overall worry/harm burden or to a single original survey feature. The clusters are then described using original non-imputed response distributions and cluster-vs-outside distributional contrasts.

The Week 3 output remains a **set of reproducible candidate resolutions with transparent simplicity and interpretation diagnostics**, not a final declaration of Objective 1. Week 4 will test alternative scaling/feature geometries, clustering algorithms, imputation choices, and feature/domain sensitivity before the final profile solution is chosen. The held-out-country respondent data remain outside model development.
